# Distributed PPO — Colab learner + GCP CPU actor + GCS transport (T=10)

Colab is the **learner / control plane**; one GCP `e2-medium` VM is the **rollout actor**;
GCS is the transport. Each iter Colab pushes a **<1 MB head-delta**; the VM's poll-daemon
rolls out 16 self-play games at **T=10** and pushes shards + logs back; Colab pulls them,
runs the PPO update, and pushes the next head-delta.

- **rollout_worker.py** (VM): applies the head-delta onto the resident frozen base, rolls
  out, uploads `shard_*.pt` + `manifest.json` + `progress.json` + `rollout.log` + `metrics.json` + `_DONE`.
- **vm_daemon.py** (VM): watches `heads/` for the newest `policy_vK.heads.pt` → runs rollout_worker.
- **learner_step.py** (Colab GPU): pulls+verifies shards → `episodes_to_ppo` → `ppo_update_local`
  → writes full `policy_v{K+1}.pt` **and** `policy_v{K+1}.heads.pt`, appends `train_log.jsonl`.

> **Run order:** §0–§4 stage + push `policy_v0`; **§5** provisions the e2-medium VM + launches the
> poll-daemon; **§6** is the control loop (push head-delta → daemon rolls out → learner_step); **§7** inspects.

## 0. Config

In [ ]:
import time
# gcloud COMPUTE needs a project set explicitly — Colab's auth.authenticate_user()
# authenticates but sets NO default project (gcloud storage works without one,
# gcloud compute does not). This is the project that owns the bucket + VM.
PROJECT = 'analog-receiver-489214-e9'
BUCKET = 'gs://orbit-wars-shipping'
RUN_ID = 'ppo_dist_' + time.strftime('%Y%m%d-%H%M%S')   # set once; keep stable across a run
PREFIX = f'{BUCKET}/ppo/{RUN_ID}'

HISTORY_WINDOW = 10           # T=10 rollout window
ITERS          = 10
EPISODES       = 16           # self-play games per iter

# --- frozen base (staged once on the VM AND on Colab) ---
# Actor: the L3/L4 PairHead supervised ckpt (conditioner_n_layers>=3, head_n_layers>=3).
BASELINE_RUN = 'L3L4_T10_film3_head3_d256_b256_20ep_lr0.0001_20260529-084115'
ENTITY_CKPT  = f'{BUCKET}/entity/runs/{BASELINE_RUN}/entity_encoder_best.pt'
# PPO critic = the value head trained INSIDE PPOActorCritic (no separate ckpt).
# The L3L4 actor has no PlayerConsolidator, so player_state is unavailable and
# the critic falls back to the glob value_head (allow_debug_glob_critic). To use
# the stronger post-L2 player_state critic, point BASELINE_RUN at a
# consolidator-equipped actor (none exists yet with L3/L4).
# BC anchor: the pair cache must match the rollout T window. '' disables BC
# (recommended until a T=10 pair cache exists; the default T6 cache mismatches).
PAIR_CACHE_PREFIX = ''               # '' = no BC; else a T=10 pair-cache prefix

# --- VM ---
VM_NAME = 'orbit-wars-ppo-actor'
VM_ZONE = 'us-central1-b'
VM_ID   = VM_NAME
SSH_USER = 'ppo'   # gcloud compute ssh user — MUST be non-root (Colab runs as
                   # root, and GCE does not provision root SSH -> Permission denied)

# --- PPO hyperparams ---
MINIBATCH=256; EPOCHS=3; CLIP=0.10; TARGET_KL=0.01; LR_HEADS=1e-4
VALUE_COEF=0.5; ENT_COEF=0.01; SIGMA=0.35; BC_COEF=0.05
print('RUN_ID', RUN_ID, '\nPREFIX', PREFIX, '\nT', HISTORY_WINDOW, 'iters', ITERS, 'episodes', EPISODES)

## 1. Authenticate

In [ ]:
from google.colab import auth
auth.authenticate_user()
import subprocess
# REQUIRED for gcloud compute (create/ssh) — without it you get
# "The required property [project] is not currently set".
subprocess.run(['gcloud','config','set','project',PROJECT], check=True)
print('project set:', PROJECT)
def gcs(*a): return subprocess.run(['gcloud','storage',*a], check=True, capture_output=True, text=True)
def gcs_exists(url):
    try: gcs('objects','describe',url,'--format=value(size)'); return True
    except subprocess.CalledProcessError: return False
print('authed:', BUCKET)

## 2. Stage code + weights + base ckpts on Colab (for the learner)

In [ ]:
import os, shutil, sys
# kaggle_environments is needed: importing the agents package pulls in agents
# that import the env at module load (the learner imports the agents package).
subprocess.run([sys.executable,'-m','pip','install','-q','kaggle_environments','psutil'], check=True)
from pathlib import Path
WORK = Path('/content/orbit-wars'); WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK)
for rel in ('agents','scripts','ckpts'): shutil.rmtree(WORK/rel, ignore_errors=True)
# code+weights live under the entity/ prefix (weights.tgz there is the FLAT
# ./{planet,fleet,comet}_*_best.pt bundle the staging below expects; the root
# weights.tgz is a different nested layout with no comet).
gcs('cp', f'{BUCKET}/entity/code.tgz', '.');    subprocess.run(['tar','xzf','code.tgz'], check=True)
gcs('cp', f'{BUCKET}/entity/weights.tgz', '.'); subprocess.run(['tar','xzf','weights.tgz'], check=True)
PLANET_DIR=WORK/'ckpts/planet'; FLEET_DIR=WORK/'ckpts/fleet'; COMET_DIR=WORK/'ckpts/comet'; ENT_DIR=WORK/'ckpts/entity'
for d in (PLANET_DIR,FLEET_DIR,COMET_DIR,ENT_DIR): d.mkdir(parents=True, exist_ok=True)
for src,dst in (('planet_encoder_best.pt',PLANET_DIR),('fleet_encoder_best.pt',FLEET_DIR),('comet_past_best.pt',COMET_DIR)):
    if (WORK/src).exists(): shutil.copy(WORK/src, dst/src)
ENTITY_LOCAL=str(ENT_DIR/'entity_encoder_best.pt')
gcs('cp', ENTITY_CKPT, ENTITY_LOCAL)
# import sanity + reset module cache
import sys
for m in [m for m in sys.modules if m.startswith('agents')]: del sys.modules[m]
import agents; from agents.transformer_v2.ppo import shards
print('staged; agents at', agents.__file__)

## 3. Stage BC pair cache (optional) — skip if PAIR_CACHE_PREFIX == ''

In [ ]:
PAIR_CACHE_LOCAL = ''
if PAIR_CACHE_PREFIX:
    # single-object pull (use the chunked pull from the other PPO notebook for the 13 GB cache)
    PAIR_CACHE_LOCAL = str(WORK/'pair_cache.pt')
    gcs('cp', f'{BUCKET}/entity/{PAIR_CACHE_PREFIX}.pt', PAIR_CACHE_LOCAL)
    print('pair cache:', PAIR_CACHE_LOCAL, round(Path(PAIR_CACHE_LOCAL).stat().st_size/1024**3,1),'GB')
else:
    print('BC anchor disabled')

## 4. Build + push `policy_v0` (full ckpt + head-delta) + config.json

In [ ]:
import torch, json
from agents.transformer_v2.ppo.smoke import load_supervised
from agents.transformer_v2.ppo.actor_critic import PPOActorCritic

entity_model, fleet_enc, planet_enc, comet_enc, cfg = load_supervised(
    Path(ENTITY_LOCAL), 'cpu', planet_run_dir=PLANET_DIR, fleet_run_dir=FLEET_DIR, comet_run_dir=COMET_DIR)
has_cons = getattr(entity_model, 'consolidator', None) is not None
print('actor PlayerConsolidator:', has_cons, '-> critic =', 'player_state' if has_cons else 'glob value_head')
# No separate critic ckpt: the value head trains inside PPO (matches smoke.py).
policy = PPOActorCritic(entity_model, sigma=SIGMA, allow_debug_glob_critic=True)
breakdown = policy.freeze_for_phase(0)
print('freeze_for_phase(0) trainable groups:', breakdown)

BASE_VERSION = shards.sha256_file(ENTITY_LOCAL)[:12]    # frozen-backbone id (entity ckpt sha)
torch.save({'policy': policy.state_dict(), 'value_hidden': policy.value_head[0].out_features,
            'sigma': SIGMA, 'iter': 0}, 'policy_v0.pt')
dinfo = shards.save_trainable_delta(policy, 'policy_v0.heads.pt', base_version=BASE_VERSION)
gcs('cp','policy_v0.pt',       f'{PREFIX}/checkpoints/policy_v0.pt')
gcs('cp','policy_v0.heads.pt', f'{PREFIX}/heads/policy_v0.heads.pt')
cfg_json = {'run_id':RUN_ID,'history_window':HISTORY_WINDOW,'base_version':BASE_VERSION,
            'action_contract':shards.POLICY_ACTION_CONTRACT,'episodes':EPISODES,'entity_ckpt':ENTITY_CKPT}
Path('config.json').write_text(json.dumps(cfg_json, indent=2)); gcs('cp','config.json',f'{PREFIX}/config.json')
print('pushed policy_v0 (full + heads delta', dinfo, ') base_version', BASE_VERSION)

## 5. Provision the e2-medium VM + launch the poll-daemon

One-time: create the VM, stage the frozen base, and start `vm_daemon.sh` under nohup. The daemon
then auto-rolls-out whenever a new `heads/policy_vK.heads.pt` appears. (If the VM already exists,
the create errors harmlessly — ignore it.)

In [ ]:
import time
def vm_running():
    r = subprocess.run(['gcloud','compute','instances','describe',VM_NAME,'--zone',VM_ZONE,
                        '--format=value(status)'], capture_output=True, text=True)
    return r.stdout.strip() == 'RUNNING'

# 1. create WITH RETRY + verify RUNNING. The old cell used check=False and
#    swallowed create failures (a transient zone-capacity error leaves NO VM, so
#    every later SSH fails). Retry, and surface a real quota/billing/org error.
for i in range(4):
    if vm_running():
        print('VM already RUNNING'); break
    cr = subprocess.run(['gcloud','compute','instances','create',VM_NAME,'--zone',VM_ZONE,
        '--machine-type','e2-medium','--image-family','debian-12','--image-project','debian-cloud',
        '--boot-disk-size','30GB','--scopes','storage-rw'], capture_output=True, text=True)
    print((cr.stdout or '') + (cr.stderr or ''))
    if cr.returncode == 0 or 'already exists' in (cr.stderr or ''):
        break
    print(f'[create attempt {i+1}/4] rc={cr.returncode}; retry in 30s ...', flush=True); time.sleep(30)
else:
    raise RuntimeError('VM create failed after 4 tries — see error above '
                       '(quota / billing / org policy / zone capacity; try another --zone).')

def ssh(cmd, *, tries=1, wait=20, label=''):
    # gcloud compute ssh with captured output + retry. The FIRST ssh to a fresh
    # VM often fails (key propagation + boot take ~30-90s); retry heals it.
    for i in range(tries):
        r = subprocess.run(['gcloud','compute','ssh',f'{SSH_USER}@{VM_NAME}','--zone',VM_ZONE,'--quiet',
                            '--command',cmd], capture_output=True, text=True)
        if r.stdout: print(r.stdout[-4000:])
        if r.returncode == 0:
            return r
        print(f"[ssh {label} attempt {i+1}/{tries} rc={r.returncode}]\n{(r.stderr or '')[-1500:]}", flush=True)
        if i < tries-1:
            time.sleep(wait)
    raise RuntimeError(f"ssh {label} failed after {tries} tries (real error in stderr above). "
                       f"Common causes: VM still booting (re-run), missing default-allow-ssh "
                       f"firewall, or IAM 'compute.instances.setMetadata' for key propagation.")

# 2. wait for SSH to come up (up to ~3 min of key-propagation/boot)
print('waiting for VM SSH (key propagation + boot can take a minute) ...', flush=True)
ssh('echo VM_READY && nproc && free -m | head -2', tries=9, wait=20, label='readiness')

# 3. bootstrap (apt + pip torch + stage — a few minutes) then launch the daemon
boot = (f"cd ~ && rm -rf orbit-wars && mkdir orbit-wars && cd orbit-wars && "
        f"gcloud storage cp {BUCKET}/entity/code.tgz . && tar xzf code.tgz && "
        f"BUCKET={BUCKET} ENTITY_CKPT={ENTITY_CKPT} bash scripts/ppo_vm_bootstrap.sh && "
        f"RUN_ID={RUN_ID} BASE_VERSION={BASE_VERSION} VM_ID={VM_ID} "
        f"HISTORY_WINDOW={HISTORY_WINDOW} EPISODES={EPISODES} bash scripts/ppo_vm_daemon.sh")
print('bootstrapping + launching daemon (a few minutes; full log captured) ...', flush=True)
ssh(boot, tries=1, label='bootstrap')
print('VM bootstrapped + daemon launched. It rolls out the NEWEST pending head-delta on GCS.')

## 6. Control loop — push head-delta → wait for `_DONE` (tail VM progress) → learner_step

In [ ]:
import time, json
def wait_done(k, timeout=5400, poll=20):
    done=f'{PREFIX}/rollouts/v{k}/{VM_ID}/_DONE'; prog=f'{PREFIX}/rollouts/v{k}/{VM_ID}/progress.json'
    t0=time.time(); last=None
    while time.time()-t0 < timeout:
        if gcs_exists(done): print(f'[vm] iter {k} DONE'); return
        p = shards.read_progress(prog)
        if p and (p.get('cur_seed'),p.get('ep_done'))!=last:
            last=(p.get('cur_seed'),p.get('ep_done'))
            print(f"[vm] iter {k} ep {p.get('ep_done')}/{p.get('ep_total')} state={p.get('state')} "
                  f"rss={p.get('rss_mb')}/{p.get('total_mb')}MB peak={p.get('peak_rss_mb')}MB "
                  f"elapsed {p.get('elapsed_s')}s eta {p.get('eta_s')}s")
        time.sleep(poll)
    raise TimeoutError(f'rollout v{k} timed out after {timeout}s')

DEV = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
import re
def latest_policy_version():
    out = subprocess.run(['gcloud','storage','ls',f'{PREFIX}/checkpoints/'],capture_output=True,text=True).stdout
    vs = [int(m.group(1)) for m in re.finditer(r'policy_v(\d+)\.pt', out)]
    return max(vs) if vs else 0
START_K = latest_policy_version()   # newest policy_vK on GCS (v0 fresh; resumes a partial run cleanly)
print('control loop starts at K =', START_K, '(daemon rolls out the newest pending head-delta)')
for K in range(START_K, ITERS):
    print(f'=== iter {K} ===')
    wait_done(K)                                   # daemon rolls out v{K} (head-delta already on GCS)
    cmd = ['python','-u','-m','agents.transformer_v2.ppo.learner_step',
       '--policy-ckpt',f'{PREFIX}/checkpoints/policy_v{K}.pt','--shards',f'{PREFIX}/rollouts/v{K}/',
       '--out-ckpt',f'{PREFIX}/checkpoints/policy_v{K+1}.pt','--out-heads',f'{PREFIX}/heads/policy_v{K+1}.heads.pt',
       '--train-log',f'{PREFIX}/train_log.jsonl','--policy-version',str(K),'--base-version',BASE_VERSION,'--device',DEV,
       '--ckpt',ENTITY_LOCAL,
       '--fleet-run-dir',str(FLEET_DIR),'--planet-run-dir',str(PLANET_DIR),'--comet-run-dir',str(COMET_DIR),
       '--minibatch-size',str(MINIBATCH),'--epochs',str(EPOCHS),'--clip',str(CLIP),'--target-kl',str(TARGET_KL),
       '--lr-heads',str(LR_HEADS),'--value-coef',str(VALUE_COEF),'--ent-coef',str(ENT_COEF),'--sigma',str(SIGMA)]
    if PAIR_CACHE_LOCAL: cmd += ['--pair-cache-path',PAIR_CACHE_LOCAL,'--bc-coef',str(BC_COEF)]
    subprocess.run(cmd, check=True)               # pushes policy_v{K+1} head-delta -> daemon rolls out next
print('loop complete')

## 7. Inspect — train log + a VM rollout log + RAM headroom

In [ ]:
import json
print('--- train_log.jsonl ---')
try:
    raw = subprocess.run(['gcloud','storage','cat',f'{PREFIX}/train_log.jsonl'],capture_output=True,text=True).stdout
    for ln in raw.splitlines():
        r=json.loads(ln); print(f"iter {r['iter']}: winrate={r.get('winrate')} kl={r.get('avg_kl')} "
              f"vloss={r.get('value_loss')} ent={r.get('entropy')} steps={r.get('n_steps')} wall={r.get('wall_s')}s")
except Exception as e: print('no train_log yet', e)
print('--- v0 metrics (RAM headroom — must be < 4 GB on e2-medium) ---')
m = shards.read_progress(f'{PREFIX}/rollouts/v0/{VM_ID}/metrics.json')
if m: print('peak_rss_mb', m.get('peak_rss_mb'),'mean_step_s', m.get('mean_step_s'),'wall_s', m.get('total_wall_s'),'wins', m.get('n_wins'))
print('--- tail rollout.log ---')
try: print(subprocess.run(['gcloud','storage','cat',f'{PREFIX}/rollouts/v0/{VM_ID}/rollout.log'],capture_output=True,text=True).stdout[-2000:])
except Exception as e: print(e)